<img src=https://audiovisuales.icesi.edu.co/assets/custom/images/ICESI_logo_prin_descriptor_RGB_POSITIVO_0924.jpg width=200>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juanestebancg2806/sugarcane-yield-quality-prediction/blob/main/03_comparacion_modelos_suertes.ipynb)

# **Proyecto Caña de Azúcar**
## Modelado y generación de insights para el TCH y la sacarosa en el Ingenio Providencia


--------------------------------------------------------------------

**Aprendizaje Automático II - Proyecto final**

**Profesor:** Anibal Sosa

**Integrantes:**

*   Juan David Martínez Legarda
*   Daniel Velasco López
*   Juan Esteban Cardona García
*   Katherin Adriana Camargo Cetina

Fecha: 22/11/2026

________________________________________________________________________

# Histórico de suertes: comparación de modelos de regresión

En este notebook se construyen y comparan modelos para anticipar el tonelaje (`TCH`, toneladas de caña por hectárea) y la calidad (`%Sac.Caña`, porcentaje de sacarosa) de cada suerte del Ingenio Providencia en el momento de programar su corte. Varios predictores, como la edad del cultivo a la cosecha y la lluvia de los últimos meses del ciclo, solo se conocen completos cerca de la fecha de corte. Por eso los modelos describen ese escenario y no un pronóstico con meses de anticipación.

El recorrido va de lo simple a lo complejo: una media constante como piso, la regresión lineal múltiple como modelo de referencia interpretable, Ridge y Lasso como chequeo de regularización, y Random Forest y XGBoost como modelos avanzados.

Para que la comparación sea justa, todos los modelos comparten el preprocesamiento, la partición en entrenamiento y prueba, los pliegues de validación cruzada y el criterio de selección. Así, una diferencia de desempeño se puede atribuir al modelo y no a la forma de evaluarlo.

**Criterios generales**

- La partición 80/20 se hace al inicio y se agrupa por hacienda: ninguna finca aparece a la vez en entrenamiento y en prueba.
- Los hiperparámetros y la selección de modelos se deciden con validación cruzada de 5 pliegues sobre el entrenamiento, con la misma agrupación. Todos los modelos se evalúan en los mismos pliegues y con un número comparable de configuraciones.
- La imputación, la agrupación de categorías raras y la codificación se estiman dentro de un `Pipeline`, de modo que en cada pliegue solo usan los datos de entrenamiento de ese pliegue.
- El conjunto de prueba se usa una sola vez, al final, con todos los modelos ya elegidos.
- Las métricas son R², RMSE y MAE. El RMSE es el criterio de selección, porque penaliza más los errores grandes; el MAE se lee en las unidades del target (t/ha o puntos de sacarosa).

**Estructura**

1. Carga del histórico limpio
2. Preprocesamiento
3. Partición en entrenamiento y prueba
4. Modelos de referencia: media constante, regresión lineal múltiple, Ridge y Lasso
5. Modelos avanzados: Random Forest y XGBoost
6. Comparación en validación cruzada y elección del modelo
7. Evaluación en el conjunto de prueba
8. Interpretabilidad
9. Conclusiones

## Punto de partida: qué dejó el análisis exploratorio

El EDA y la preparación de los datos se hicieron en `01_eda_suertes.ipynb`. A continuación se resume lo que se necesita de ese trabajo para el modelado.

**Los datos.** `HISTORICO_SUERTES.xlsx` tiene 21 027 cosechas y 85 columnas, entre enero de 2017 y julio de 2024. Cada fila es una suerte (un lote, identificado por `Hacienda` y `Suerte`) cosechada en un mes (`Período`, en formato AAAAMM). Un mismo lote aparece varias veces en el histórico, una por cada corte. La población es amplia: incluye lotes madurados y sin madurante, cosecha mecánica y manual, manejo convencional y orgánico, y 58 variedades.

**Los dos targets**

| | `TCH` | `%Sac.Caña` |
|---|---|---|
| Qué mide | Productividad: toneladas de caña por hectárea | Calidad: porcentaje de sacarosa en la caña |
| Distribución | Casi simétrica, con media cercana a 130 t/ha | Concentrada, con media de 12.3 % y rango de 7.1 a 18.4 % |
| Registros que se excluyen | 355 lotes de destino molino con TCH menor que 60 t/ha, que por criterio de negocio se consideran atípicos. Los valores altos (hasta 401 t/ha) corresponden a lotes de área pequeña y se conservan | Los 449 registros sin valor medido, en su mayoría lotes de semilla que no pasan por el laboratorio. El target no se imputa |

La correlación entre los dos targets es de −0.17: más caña no implica más azúcar. Por eso se modelan como dos problemas independientes, con el mismo conjunto de predictores.

**Lo que quedó fuera de los predictores**

| Columnas | Motivo |
|---|---|
| Fertilizantes y variables de la estación meteorológica (temperatura, humedad, radiación, evaporación) | 70 % o más de nulos. La estación no tiene registros en la mayor parte del histórico, y la fertilización casi no se registró |
| `Nombre`, `Hacienda`, `Suerte` y los códigos de texto duplicados (`Cod.Estado`, `Cod. T.Cultivo`) | Identifican el registro, no describen el cultivo |
| Toneladas, azúcar por hectárea, `TCHM`, `Rdto`, mediciones de laboratorio (`%ATR`, Brix, Pureza, etc.) y `Vejez` | Se miden después del corte o se derivan del propio target. Usarlas sería fuga de información |
| `Lluvias Ciclo` | Es la suma exacta de las cuatro ventanas de lluvia del ciclo |
| Edad desde la siembra | Repite el número de corte |
| `Area Neta` | El TCH ya está expresado por hectárea |
| `Producto` | Su información ya está en la dosis y las semanas de madurante |
| `Cerca de` (municipio más cercano) | Repite en gran parte la zona |
| `Cosechó`, `D.S.`, `DDUlt.Riego` y las fechas (`F.Siembra`, `F.Ult.Corte`, `Ult.Riego`, `Fec.Madur.`) | Casi constantes, con muchos nulos o ya resumidas en la edad del ciclo y las semanas de madurante |
| `Sac.Caña Precosecha` y `Edad.Precosecha` | Muestreo tomado antes del corte. Su uso como predictor se revisa en la sección 1.1 |

**Los predictores y su justificación**

| Grupo | Variables | Por qué entran | Tratamiento de los vacíos y niveles raros |
|---|---|---|---|
| Ciclo del cultivo | `Edad Ult Cos` (meses del ciclo actual), `Cod.Estado #` (número de corte) | La edad a la cosecha y el envejecimiento de la soca determinan el tonelaje | Sin nulos |
| Maduración | `Dosis Madurante` (L/ha), `Semanas mad.` (desde la aplicación hasta el corte) | El madurante es la principal palanca de manejo sobre la sacarosa | Dosis sin dato = no se aplicó madurante; semanas = 0 en lotes sin madurante |
| Lluvia | Ventanas de 0–3, 3–6, 6–9 y 9 meses a cosecha, lluvia de los 2 meses previos, `lluvia_sin_dato` | El efecto de la lluvia depende de la fase del ciclo | Los ceros son, en su mayoría, falta de medición: se conservan y se marcan con la bandera |
| Riego | `Num.Riegos`, `M3 Riego` | Complementa el agua de lluvia | Sin dato de riegos y sin volumen de agua = no se regó |
| Ubicación | `Zona`, `Suelo`, `Dist Km` | La zona agroecológica y el suelo definen el potencial productivo | `Suelo` sin dato es una categoría propia (fincas sin estudio de suelos). Distancia imputada por zona |
| Manejo | `Variedad`, `T.Corte`, `Tipo Quema`, `Cultivo` (normal u orgánico), `Tenencia`, `Destino 1=Semilla` | Cada variedad tiene su propio potencial y curva de maduración; el manejo orgánico no usa madurante químico | Variedades, suelos y tenencias poco frecuentes se agrupan en `OTRAS` |
| Tiempo | `Año`, `Mes` | Tendencia entre años y estacionalidad de la cosecha | Se derivan de `Período` |

**Relaciones que encontró el EDA**

- El TCH baja de unas 148 t/ha en plantilla a unas 115 t/ha hacia el corte 9 o 10, y desde ahí se aplana. Es un codo, no una recta.
- La edad del ciclo se asocia positivamente con el TCH (correlación de 0.30) y la dosis de madurante con la sacarosa (0.29).
- La lluvia de los primeros 3 meses se asocia con menos tonelaje, y la de los 2 meses previos al corte con menos sacarosa: una correlación de Spearman de −0.23, coherente con la dilución del azúcar al final del ciclo.
- El efecto de la lluvia temprana sobre el TCH cambia de signo según la variedad: hay interacción entre clima y variedad.
- `Dist Km` tiene una correlación lineal casi nula con el TCH, pero una relación en forma de curva por tramos de distancia. Además, toma un único valor por hacienda, así que en parte funciona como identificador de la finca.
- El cultivo orgánico tiene un tonelaje parecido al convencional, pero menos sacarosa (10.6 % frente a 12.4 %).

**Limitaciones de los datos**

- Los datos de la estación meteorológica no cubren la mayor parte del histórico, y la fertilización casi no se registró. El clima se representa solo con la lluvia del lote.
- Más de un tercio de los registros no tiene lluvia medida.
- Algunas decisiones del EDA (selección de predictores, umbrales de agrupación, criterio de atípicos) se tomaron mirando todo el histórico. No estiman ningún estadístico que usen los modelos, pero conviene tenerlas presentes al leer el desempeño en prueba.

In [1]:
# Librerías generales del notebook. Los módulos de scikit-learn se importan en la sección donde se usan.
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import statsmodels
import statsmodels.api as sm
import xgboost

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", None)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11

%matplotlib inline

# Versiones usadas, para la reproducibilidad de los resultados
print(f"pandas {pd.__version__} | numpy {np.__version__} | scikit-learn {sklearn.__version__} | "
      f"statsmodels {statsmodels.__version__} | xgboost {xgboost.__version__}")
      

pandas 3.0.6 | numpy 2.5.4 | scikit-learn 1.9.1 | statsmodels 0.15.0 | xgboost 3.4.1


## 1. Carga del histórico limpio

La última celda de `01_eda_suertes.ipynb` guarda el histórico en `datos/suertes_limpio.parquet`. El archivo trae solo transformaciones que dan el mismo resultado en cada fila sin importar las demás: tipos de dato, textos de vacío convertidos a nulos, etiquetas unificadas y dos marcas de fila creadas en el EDA:

- **`excluir_tch`** vale `True` en los lotes de destino molino con TCH menor que 60 t/ha. Por criterio de negocio, un tonelaje tan bajo en un lote comercial es atípico. Esos 355 registros no se borran del archivo: solo se excluyen del modelo de `TCH`, y siguen en el de `%Sac.Caña` porque su sacarosa sí está medida.
- **`lluvia_sin_dato`** vale `True` cuando la lluvia de todo el ciclo es cero. La precipitación media del valle geográfico del río Cauca está entre 850 y 2500 mm al año, y la cuenca tiene dos temporadas lluviosas, de marzo a mayo y de septiembre a noviembre [1]. Un ciclo completo sin lluvia no es una sequía sino un registro faltante, y así se trataron esos ceros en el EDA. La marca entra como predictor, para que el modelo distinga la lluvia medida de la no registrada.

El archivo no trae ninguna imputación, porque los estadísticos para imputar deben calcularse solo con el conjunto de entrenamiento.

Antes de modelar se verifica que el archivo conserve lo establecido en el EDA:

- 21 027 cosechas, sin repetir la combinación de lote y periodo.
- Los nulos originales de los predictores que se imputan más adelante. Si alguno cambia, el archivo llegó imputado.
- Las 355 filas con `excluir_tch` y las 7739 con `lluvia_sin_dato`.
- 20 672 registros válidos para `TCH` y 20 578 para `%Sac.Caña`.
- Las columnas `Hacienda`, `Suerte` y `Período`, que no son predictores pero se necesitan para agrupar la partición.

Si alguna verificación falla, el notebook se detiene.

**Referencias**

[1] J. D. Mera Franco y C. Y. Soto Chávez, "Análisis espaciotemporal de las tendencias de precipitación y temperatura en la cuenca del río Cauca y el Pacífico vallecaucano," Corporación Autónoma Regional del Valle del Cauca (CVC), Santiago de Cali, Colombia, Informe técnico, dic. 2024. [En línea]. Disponible: https://ecopedia.cvc.gov.co/sites/default/files/archivosAdjuntos/Informe%20Tendencias%20Precipitacion%20Temperatura.pdf

In [2]:
# 1. Carga del histórico limpio (salida de 01_eda_suertes.ipynb) y verificación de sus propiedades
RANDOM_STATE = 42  # semilla común a la partición, los pliegues y los modelos (la misma de AA1)
RUTA_SUERTES = Path("datos/suertes_limpio.parquet")

# Propiedades del archivo establecidas en 01_eda_suertes.ipynb
FILAS_ESPERADAS = 21_027
NULOS_ORIGINALES = {"Semanas mad.": 9_775, "Num.Riegos": 12_589, "Dosis Madurante": 104, "Suelo": 3_758}
FILAS_EXCLUIR_TCH = 355
FILAS_LLUVIA_SIN_DATO = 7_739
FILAS_VALIDAS_POR_TARGET = {"TCH": 20_672, "%Sac.Caña": 20_578}
COLUMNAS_REQUERIDAS = [
    "Hacienda", "Suerte", "Período", "TCH", "%Sac.Caña", "excluir_tch", "lluvia_sin_dato",
    "Lluvias 0 -3", "Lluvias tres a seis", "Lluvias seis a nueve", "Luvias 9 -FC", "Lluvias (2 Meses Ant.)",
]


def formato_miles(numero):
    """Entero con espacio como separador de miles a partir de cinco cifras (21 027), según la RAE."""
    return f"{numero:,}".replace(",", " ") if abs(numero) >= 10_000 else str(numero)


def filas_validas(df, target):
    """Máscara de los registros con los que se modela cada target, sin imputarlo.

    TCH excluye los lotes marcados con excluir_tch (TCH < 60 t/ha de destino molino) y
    %Sac.Caña conserva solo los registros con valor medido.
    """
    if target == "TCH":
        return ~df["excluir_tch"]
    return df["%Sac.Caña"].notna()


def verificar_historico(df):
    """Detiene el notebook si el archivo no conserva las propiedades establecidas en el EDA."""
    faltantes = [col for col in COLUMNAS_REQUERIDAS if col not in df.columns]
    assert not faltantes, f"Faltan columnas: {faltantes}"
    assert len(df) == FILAS_ESPERADAS, (
        f"Se esperaban {formato_miles(FILAS_ESPERADAS)} filas y llegaron {formato_miles(len(df))}"
    )
    assert not df.duplicated(["Hacienda", "Suerte", "Período"]).any(), "La combinación lote + periodo se repite"

    # Si algún nulo cambió, el archivo llegó imputado
    for col, nulos_esperados in NULOS_ORIGINALES.items():
        nulos = df[col].isna().sum()
        assert nulos == nulos_esperados, (
            f"{col}: {formato_miles(nulos)} nulos, se esperaban {formato_miles(nulos_esperados)}"
        )

    assert df["excluir_tch"].sum() == FILAS_EXCLUIR_TCH, "Cambió el número de filas con excluir_tch"
    assert df["lluvia_sin_dato"].sum() == FILAS_LLUVIA_SIN_DATO, "Cambió el número de filas con lluvia_sin_dato"
    for target, filas_esperadas in FILAS_VALIDAS_POR_TARGET.items():
        assert filas_validas(df, target).sum() == filas_esperadas, f"Cambió el número de registros válidos de {target}"


if not RUTA_SUERTES.exists():
    raise FileNotFoundError(
        f"No existe {RUTA_SUERTES}. Ejecutar antes 01_eda_suertes.ipynb: su última celda genera el archivo."
    )

df_suertes = pd.read_parquet(RUTA_SUERTES)
verificar_historico(df_suertes)

n_lotes = df_suertes.groupby(["Hacienda", "Suerte"]).ngroups
print(f"Histórico limpio: {formato_miles(df_suertes.shape[0])} filas × {df_suertes.shape[1]} columnas")
print(f"Haciendas: {formato_miles(df_suertes['Hacienda'].nunique())} | lotes: {formato_miles(n_lotes)} | "
      f"cortes por lote: {len(df_suertes) / n_lotes:.1f} en promedio")
print(f"Periodos: {df_suertes['Período'].nunique()} (de {df_suertes['Período'].min()} a {df_suertes['Período'].max()})")
for target in FILAS_VALIDAS_POR_TARGET:
    print(f"Registros válidos de {target}: {formato_miles(filas_validas(df_suertes, target).sum())}")

Histórico limpio: 21 027 filas × 88 columnas
Haciendas: 926 | lotes: 5799 | cortes por lote: 3.6 en promedio
Periodos: 91 (de 201701 a 202407)
Registros válidos de TCH: 20 672
Registros válidos de %Sac.Caña: 20 578


### 1.1 Sacarosa precosecha

En el EDA, `Sac.Caña Precosecha` se identificó como una medición tomada antes del corte y no como fuga de información, pero quedó fuera del conjunto final de predictores sin un análisis específico. Es la única variable del histórico que mide directamente el contenido de sacarosa del lote antes de cosecharlo, así que se revisa si puede entrar al modelo. Para eso debe cumplir tres condiciones:

- **Cobertura.** Que los registros con muestreo no se concentren en un solo periodo, como ocurre con la estación meteorológica.
- **Momento de la medición.** Que la muestra se tome antes del corte. Se compara la edad del cultivo al muestrear (`Edad.Precosecha`) con la edad a la cosecha (`Edad Ult Cos`).
- **Relación con el target.** Que tenga una asociación con `%Sac.Caña` que justifique incluirla, sin ser casi idéntica a ella. Una correlación cercana a 1 indicaría que la medición no se distingue del resultado del corte.

In [3]:
# 1.1 Sacarosa precosecha: cobertura, momento de la medición y relación con el target
COLUMNA_PRECOSECHA = "Sac.Caña Precosecha"
RANGO_PLAUSIBLE_SACAROSA = (df_suertes["%Sac.Caña"].min(), df_suertes["%Sac.Caña"].max())  # rango del target

precosecha = df_suertes[COLUMNA_PRECOSECHA]
# Una sacarosa de 0 % no es posible: los ceros son faltantes disfrazados, como en la lluvia
muestra_valida = precosecha.notna() & (precosecha > 0)

print(f"Nulos: {precosecha.isna().mean():.1%} | ceros: {(precosecha == 0).mean():.1%} | "
      f"valores mayores que 0: {muestra_valida.mean():.1%} ({formato_miles(muestra_valida.sum())} registros)")

# Distribución de los valores válidos frente al rango del target
valores_validos = precosecha[muestra_valida]
fuera_de_rango = ~valores_validos.between(*RANGO_PLAUSIBLE_SACAROSA)
print(f"\nValores mayores que 0: mediana {valores_validos.median():.2f} | "
      f"p5 {valores_validos.quantile(0.05):.2f} | p95 {valores_validos.quantile(0.95):.2f} | "
      f"máximo {valores_validos.max():.2f}")
print(f"Fuera del rango de %Sac.Caña ({RANGO_PLAUSIBLE_SACAROSA[0]:.1f} a {RANGO_PLAUSIBLE_SACAROSA[1]:.1f}): "
      f"{fuera_de_rango.mean():.1%}")

# Cobertura de los valores válidos por año de cosecha
cobertura_por_año = (df_suertes.assign(año=df_suertes["Período"] // 100, muestra_valida=muestra_valida)
                     .groupby("año")["muestra_valida"].agg(registros="size", proporcion_valida="mean"))
print("\nProporción de registros con muestra válida por año:")
display(cobertura_por_año.round(3))

# Momento de la medición, solo en registros con muestra válida y edad de muestreo registrada
con_edad_muestreo = muestra_valida & df_suertes["Edad.Precosecha"].notna()
meses_antes_del_corte = (df_suertes.loc[con_edad_muestreo, "Edad Ult Cos"]
                         - df_suertes.loc[con_edad_muestreo, "Edad.Precosecha"])
print(f"Meses entre la muestra y el corte (n = {formato_miles(con_edad_muestreo.sum())}): "
      f"mediana {meses_antes_del_corte.median():.2f} | "
      f"muestras posteriores al corte: {(meses_antes_del_corte < 0).mean():.1%}")

# Relación con el target, solo en registros con muestra válida y sacarosa medida
pares_validos = df_suertes.loc[muestra_valida & df_suertes["%Sac.Caña"].notna(), [COLUMNA_PRECOSECHA, "%Sac.Caña"]]
print(f"\nCorrelación con %Sac.Caña en registros con muestra válida (n = {formato_miles(len(pares_validos))}): "
      f"Pearson {pares_validos.corr().iloc[0, 1]:.3f} | "
      f"Spearman {pares_validos.corr(method='spearman').iloc[0, 1]:.3f}")

Nulos: 5.2% | ceros: 51.2% | valores mayores que 0: 43.6% (9159 registros)

Valores mayores que 0: mediana 17.07 | p5 15.05 | p95 18.52 | máximo 44.78
Fuera del rango de %Sac.Caña (7.1 a 18.4): 6.5%

Proporción de registros con muestra válida por año:


,registros,proporcion_valida
año,,
2017,2517,0.785
2018,2879,0.785
2019,2852,0.788
2020,2889,0.796
2021,2504,0.143
2022,2932,0.004
2023,3022,0.003
2024,1432,0.001


Meses entre la muestra y el corte (n = 8754): mediana 0.59 | muestras posteriores al corte: 6.9%

Correlación con %Sac.Caña en registros con muestra válida (n = 9144): Pearson 0.304 | Spearman 0.371


### Conclusión de la sección 1

El archivo cumple todas las verificaciones. Tiene 21 027 filas y 88 columnas: las 85 del Excel original más tres creadas en el EDA (la edad desde la siembra, `excluir_tch` y `lluvia_sin_dato`). Los nulos de los predictores son los originales, así que toda la imputación queda por hacer dentro del modelado.

Las cosechas provienen de 926 haciendas y 5799 lotes, en 91 periodos entre enero de 2017 y julio de 2024. Cada lote aparece en promedio 3.6 veces, una por corte, y cada hacienda agrupa varios lotes. Esa estructura es la que obliga a agrupar la partición, para que el conjunto de prueba no comparta fincas con el entrenamiento.

Se modelan 20 672 cosechas para `TCH` y 20 578 para `%Sac.Caña`.

**`Sac.Caña Precosecha` queda fuera de los predictores.** Solo el 5.2 % de los registros está vacío, pero el 51.2 % tiene un cero, que no es una medición posible. Los valores válidos cubren el 43.6 % del histórico y se concentran entre 2017 y 2020 (entre el 78 % y el 80 % de los registros de cada año); bajan al 14 % en 2021 y desaparecen desde 2022. Un modelo que dependiera de esta variable no podría aplicarse a las cosechas actuales, y la ausencia de la muestra se confundiría con el año de cosecha. Además, sus valores no están en la misma escala que el target: la mediana es 17.1, cerca del extremo superior del rango de `%Sac.Caña` (7.1 a 18.4), y el 6.5 % cae fuera de ese rango. El 6.9 % de las muestras con edad registrada sería posterior al corte. Su relación con la sacarosa de la cosecha es moderada (Spearman de 0.37). El muestreo precosecha podría ser un predictor útil si el Ingenio lo registrara de forma continua y en la misma base que la sacarosa de la cosecha.

## 2. Preprocesamiento

El preprocesamiento aplica las reglas de imputación y codificación definidas en el EDA. Se implementa como un transformador de scikit-learn y no como pasos sueltos sobre el DataFrame, por dos razones:

- Dentro de un `Pipeline`, la validación cruzada y la búsqueda de hiperparámetros reajustan el transformador en cada pliegue, de modo que cada estadístico se calcula solo con los datos de entrenamiento de ese pliegue.
- El conjunto de prueba recibe exactamente las mismas transformaciones, con los valores aprendidos en el entrenamiento.

### 2.1 Reglas de negocio y transformador

Las reglas son de dos tipos:

- **Sin estado.** El resultado de cada fila no depende de las demás, así que se aplican igual en entrenamiento y en prueba: dosis de madurante sin dato → 0 (no se aplicó), semanas de madurante → 0 en lotes sin madurante, riegos sin dato → 0 cuando el volumen de agua es 0, `Período` separado en `Año` y `Mes`, y fusión de los niveles con un único registro.
- **Estimadas en el ajuste.** La mediana de `Semanas mad.` en los lotes madurados, la mediana de `Num.Riegos` en los lotes con riego, la distancia, la tenencia y el sistema de cultivo por zona (con el valor global como respaldo), los niveles de variedad, suelo y tenencia con suficientes registros para tener columna propia, y la categoría de referencia de cada variable categórica.

Cuatro decisiones adicionales:

- **`madurada`** (si el lote recibió madurante) se usa para imputar `Semanas mad.`, pero no entra como predictor. Casi repite la información de la dosis, porque una dosis de 0 equivale a no haber madurado (correlación de 0.92 en el EDA).
- **`Año` se centra en 2017**, para que el intercepto del modelo lineal corresponda al primer año del histórico y no a un año 0 que no existe en los datos.
- **La categoría de referencia** de cada variable categórica es su nivel más frecuente en el ajuste, sin contar los niveles residuales (`OTRAS`, `SIN DATO`, `NO DEF.`). Así, el coeficiente de cada indicadora se lee como la diferencia frente a la condición más común en el ingenio, y no frente a un nivel elegido por orden alfabético, como en AA1.
- **El transformador devuelve un `DataFrame` con los nombres de las columnas.** Los nombres se necesitan para leer coeficientes e importancias en las secciones 4 y 8.

In [4]:
# 2.1 Reglas de negocio y transformador de preprocesamiento del histórico de suertes
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted

AÑO_REFERENCIA = 2017  # el intercepto del modelo lineal corresponde al primer año del histórico

# Predictores que entran a los modelos. madurada no entra: casi repite a Dosis Madurante.
PREDICTORES_NUMERICOS = [
    "Edad Ult Cos", "Cod.Estado #", "Dist Km", "Dosis Madurante", "Semanas mad.",
    "Lluvias 0 -3", "Lluvias tres a seis", "Lluvias seis a nueve", "Luvias 9 -FC",
    "Lluvias (2 Meses Ant.)", "lluvia_sin_dato", "Num.Riegos", "M3 Riego",
    "Destino 1=Semilla", "Año",
]
PREDICTORES_CATEGORICOS = [
    "Zona_agrup", "Variedad_agrup", "Suelo_agrup", "T.Corte_agrup",
    "Tipo Quema", "Cultivo", "Tenencia_agrup", "Mes",
]
# Mínimo de registros para que un nivel tenga columna propia; los demás se agrupan en OTRAS
MINIMO_REGISTROS_POR_NIVEL = {"Variedad": 200, "Suelo": 300, "Tenencia": 200}
# Niveles residuales que no sirven como base de comparación de los coeficientes
NIVELES_RESIDUALES = {"OTRAS", "SIN DATO", "NO DEF."}

# Columnas crudas que necesita el preprocesamiento: predictores originales y columnas auxiliares
COLUMNAS_CRUDAS = [
    "Edad Ult Cos", "Cod.Estado #", "Dist Km", "Dosis Madurante", "Producto", "Semanas mad.",
    "Lluvias 0 -3", "Lluvias tres a seis", "Lluvias seis a nueve", "Luvias 9 -FC",
    "Lluvias (2 Meses Ant.)", "lluvia_sin_dato", "Num.Riegos", "M3 Riego", "Destino 1=Semilla",
    "Período", "Zona", "Variedad", "Suelo", "T.Corte", "Tipo Quema", "Cultivo", "Tenencia",
]


def moda(serie):
    """Valor más frecuente ignorando los nulos (NaN si la serie no tiene valores)."""
    modas = serie.dropna().mode()
    return modas.iloc[0] if len(modas) else np.nan


def aplicar_reglas_negocio(df):
    """Recodificaciones sin estado: el resultado de cada fila no depende de las demás."""
    datos = df.copy()

    # Madurante: sin dosis ni producto registrados significa que no se aplicó
    datos["madurada"] = ((datos["Dosis Madurante"].fillna(0) > 0) | datos["Producto"].notna()).astype(int)
    datos["Dosis Madurante"] = datos["Dosis Madurante"].fillna(0)
    # Semanas negativas son un error de captura; sin madurante, las semanas desde su aplicación son 0
    datos.loc[datos["Semanas mad."] < 0, "Semanas mad."] = np.nan
    sin_madurante = datos["madurada"] == 0
    datos.loc[sin_madurante, "Semanas mad."] = datos.loc[sin_madurante, "Semanas mad."].fillna(0)

    # Riego: sin número de riegos y sin volumen de agua significa que no se regó
    datos.loc[datos["Num.Riegos"].isna() & (datos["M3 Riego"] == 0), "Num.Riegos"] = 0

    # Período (AAAAMM) no es una escala: se separa en tendencia (Año) y estacionalidad (Mes)
    datos["Año"] = datos["Período"] // 100 - AÑO_REFERENCIA
    datos["Mes"] = (datos["Período"] % 100).astype(str).str.zfill(2)

    # Niveles con un único registro o sin significado propio
    datos["Zona_agrup"] = datos["Zona"].replace({"IC07": "OTRAS"})
    datos["T.Corte_agrup"] = datos["T.Corte"].replace({"SEMILLA": "OTRAS"})
    # SEMILLA no es un tipo de quema: repite la información de Destino 1=Semilla
    datos["Tipo Quema"] = datos["Tipo Quema"].replace({"SEMILLA": "VERDE"})
    return datos


def nivel_mas_frecuente(serie):
    """Nivel más frecuente que no sea residual; si todos son residuales, el más frecuente."""
    conteo = serie.value_counts()
    no_residuales = conteo[~conteo.index.isin(NIVELES_RESIDUALES)]
    return (no_residuales if len(no_residuales) else conteo).idxmax()


class PreprocesadorSuertes(BaseEstimator, TransformerMixin):
    """Imputación, agrupación de niveles poco frecuentes y codificación indicadora del histórico.

    Todo lo que se estima a partir de los datos (medianas, modas por zona, niveles que se conservan
    y categorías de referencia) se aprende en fit(). Dentro de un Pipeline, la validación cruzada
    lo recalcula en cada pliegue con los datos de entrenamiento de ese pliegue.

    Parámetros
    ----------
    referencia : {"frecuente", "alfabetica"}
        Cómo se elige la categoría de referencia de cada variable categórica: su nivel más
        frecuente no residual (por defecto) o el primero en orden alfabético (la regla de AA1).
    """

    def __init__(self, referencia="frecuente"):
        self.referencia = referencia

    def fit(self, X, y=None):
        if self.referencia not in {"frecuente", "alfabetica"}:
            raise ValueError(f"referencia debe ser 'frecuente' o 'alfabetica', no {self.referencia!r}")
        datos = aplicar_reglas_negocio(X)

        # Imputación condicionada al significado del vacío
        self.mediana_semanas_madurados_ = datos.loc[datos["madurada"] == 1, "Semanas mad."].median()
        self.mediana_riegos_con_riego_ = datos.loc[datos["M3 Riego"] > 0, "Num.Riegos"].median()
        # Atributos de la finca: se imputan con el valor típico de su zona
        self.distancia_por_zona_ = datos.groupby("Zona")["Dist Km"].median()
        self.tenencia_por_zona_ = datos.groupby("Zona")["Tenencia"].agg(moda)
        self.cultivo_por_zona_ = datos.groupby("Zona")["Cultivo"].agg(moda)
        self.distancia_global_ = datos["Dist Km"].median()
        self.tenencia_global_ = moda(datos["Tenencia"])
        self.cultivo_global_ = moda(datos["Cultivo"])
        datos = self._imputar(datos)

        # Niveles con suficientes registros para tener columna propia
        self.niveles_conservados_ = {}
        for columna, minimo in MINIMO_REGISTROS_POR_NIVEL.items():
            conteo = datos[columna].value_counts()
            self.niveles_conservados_[columna] = set(conteo[conteo >= minimo].index)
        datos = self._agrupar_niveles(datos)

        # Categoría de referencia de cada categórica. Su indicadora se elimina para evitar
        # colinealidad exacta con el intercepto.
        categoricas = datos[PREDICTORES_CATEGORICOS].astype(str)
        if self.referencia == "frecuente":
            self.niveles_referencia_ = {col: nivel_mas_frecuente(categoricas[col]) for col in PREDICTORES_CATEGORICOS}
        else:
            self.niveles_referencia_ = {col: categoricas[col].min() for col in PREDICTORES_CATEGORICOS}
        columnas_referencia = [f"{col}_{nivel}" for col, nivel in self.niveles_referencia_.items()]
        self.columnas_indicadoras_ = pd.get_dummies(categoricas, dtype=float).columns.drop(columnas_referencia)
        self.nombres_columnas_ = np.array(PREDICTORES_NUMERICOS + list(self.columnas_indicadoras_), dtype=object)
        return self

    def _imputar(self, datos):
        datos = datos.copy()
        datos["Semanas mad."] = datos["Semanas mad."].fillna(self.mediana_semanas_madurados_)
        datos["Num.Riegos"] = datos["Num.Riegos"].fillna(self.mediana_riegos_con_riego_)
        datos["Suelo"] = datos["Suelo"].fillna("SIN DATO")  # finca sin estudio de suelos: categoría propia
        datos["Dist Km"] = (datos["Dist Km"].fillna(datos["Zona"].map(self.distancia_por_zona_))
                            .fillna(self.distancia_global_))
        datos["Tenencia"] = (datos["Tenencia"].fillna(datos["Zona"].map(self.tenencia_por_zona_))
                             .fillna(self.tenencia_global_))
        datos["Cultivo"] = (datos["Cultivo"].fillna(datos["Zona"].map(self.cultivo_por_zona_))
                            .fillna(self.cultivo_global_))
        datos["Tenencia"] = datos["Tenencia"].astype("Int64").astype(str)  # código, no escala numérica
        return datos

    def _agrupar_niveles(self, datos):
        datos = datos.copy()
        for columna, conservados in self.niveles_conservados_.items():
            datos[f"{columna}_agrup"] = datos[columna].where(datos[columna].isin(conservados), "OTRAS")
        return datos

    def transform(self, X):
        check_is_fitted(self, "columnas_indicadoras_")
        datos = self._agrupar_niveles(self._imputar(aplicar_reglas_negocio(X)))
        # Reindexando con las columnas del ajuste: la referencia queda fuera y un nivel que no
        # apareció en el ajuste queda con todas sus indicadoras en 0, como la referencia.
        indicadoras = (pd.get_dummies(datos[PREDICTORES_CATEGORICOS].astype(str), dtype=float)
                       .reindex(columns=self.columnas_indicadoras_, fill_value=0.0))
        return pd.concat([datos[PREDICTORES_NUMERICOS].astype(float), indicadoras], axis=1)

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, "nombres_columnas_")
        return self.nombres_columnas_


def separar_target(df, target):
    """Registros válidos de un target: columnas crudas que usa el preprocesamiento y valores del target."""
    validos = df.loc[filas_validas(df, target)]
    return validos[COLUMNAS_CRUDAS], validos[target].astype(float)

### 2.2 Verificación del transformador

El transformador reescribe el preprocesamiento usado en AA1 (`02_modelos_regresion.ipynb`). Para confirmar que aplica las mismas reglas, se reproduce un resultado de ese notebook con la referencia alfabética que se usó entonces: el R² de validación cruzada de la regresión lineal con el preprocesamiento reajustado en cada pliegue, que fue de 0.2933 en `TCH` y 0.3408 en `%Sac.Caña`. Para obtenerlo se usan la partición aleatoria y los pliegues de AA1, solo en esta celda; la partición de este notebook se define en la sección 3.

Además se verifica que el transformador, con la referencia que se usa en el resto del notebook, no deje faltantes al transformar registros que no intervinieron en su ajuste, y se listan las categorías de referencia resultantes.

In [5]:
# 2.2 Verificación: reproducir el R² de validación cruzada del modelo lineal de AA1
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline

# Partición aleatoria y pliegues de AA1: se usan solo en esta verificación
indices_train_aa1, indices_test_aa1 = train_test_split(
    df_suertes.index, test_size=0.20, random_state=RANDOM_STATE, shuffle=True
)
df_train_aa1 = df_suertes.loc[indices_train_aa1]
pliegues_aa1 = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
R2_CV_AA1 = {"TCH": 0.2933, "%Sac.Caña": 0.3408}  # 02_modelos_regresion.ipynb, paso 23.4
TOLERANCIA_R2 = 5e-4  # los valores de AA1 están redondeados a 4 decimales

for target, r2_aa1 in R2_CV_AA1.items():
    X_verificacion, y_verificacion = separar_target(df_train_aa1, target)
    pipeline_lineal_aa1 = Pipeline([
        ("preprocesamiento", PreprocesadorSuertes(referencia="alfabetica")),  # la regla de AA1
        ("modelo", LinearRegression()),
    ])
    resultados = cross_validate(pipeline_lineal_aa1, X_verificacion, y_verificacion, cv=pliegues_aa1, scoring="r2")
    r2_cv = resultados["test_score"].mean()
    print(f"{target}: R² CV = {r2_cv:.4f} (AA1: {r2_aa1:.4f}) | n = {formato_miles(len(y_verificacion))}")
    assert abs(r2_cv - r2_aa1) < TOLERANCIA_R2, f"{target}: el preprocesamiento no reproduce el de AA1"

# Ajuste con el entrenamiento de AA1 y transformación de registros que no intervinieron en el ajuste
X_ajuste, _ = separar_target(df_train_aa1, "TCH")
preprocesador_verificacion = PreprocesadorSuertes().fit(X_ajuste)
matriz_no_vista = preprocesador_verificacion.transform(df_suertes.loc[indices_test_aa1, COLUMNAS_CRUDAS])

assert matriz_no_vista.isna().sum().sum() == 0, "La transformación dejó faltantes"
assert list(matriz_no_vista.columns) == list(preprocesador_verificacion.get_feature_names_out())

n_indicadoras = len(preprocesador_verificacion.columnas_indicadoras_)
print(f"\nMatriz de diseño: {matriz_no_vista.shape[1]} columnas "
      f"({len(PREDICTORES_NUMERICOS)} numéricas y {n_indicadoras} indicadoras), sin faltantes")
print("Categorías de referencia:")
for columna, referencia in preprocesador_verificacion.niveles_referencia_.items():
    print(f"  {columna}: {referencia}")

TCH: R² CV = 0.2933 (AA1: 0.2933) | n = 16 538
%Sac.Caña: R² CV = 0.3408 (AA1: 0.3408) | n = 16 479

Matriz de diseño: 57 columnas (15 numéricas y 42 indicadoras), sin faltantes
Categorías de referencia:
  Zona_agrup: IP05
  Variedad_agrup: CC01-1940
  Suelo_agrup: CORINTIAS
  T.Corte_agrup: MECANIZADO
  Tipo Quema: VERDE
  Cultivo: Normal
  Tenencia_agrup: 51
  Mes: 01


### Conclusión de la sección 2

Con la referencia alfabética de AA1, el transformador reproduce exactamente el R² de validación cruzada de ese notebook: 0.2933 en `TCH` y 0.3408 en `%Sac.Caña`. Las reglas de imputación y codificación son las mismas, así que los modelos de este notebook parten del preprocesamiento definido en el EDA, ahora estimado dentro de cada pliegue.

La matriz de diseño tiene 57 columnas: 15 numéricas y 42 indicadoras. No deja faltantes al transformar registros que no intervinieron en el ajuste.

Las categorías de referencia corresponden a la condición más común en el histórico: zona IP05, variedad CC01-1940, suelo Corintias, cosecha mecanizada, cosecha en verde, manejo convencional, tenencia 51 y enero como mes de cosecha. En la sección 4, cada coeficiente de una indicadora se lee como la diferencia frente a esa condición.

## 3. Partición en entrenamiento y prueba

La partición tiene que reproducir la forma en que se usaría el modelo. El Ingenio lo aplicaría a cosechas que todavía no ocurren, incluidas las de fincas cuyo historial no está en los datos de ajuste, como las de proveedores nuevos. Si el conjunto de prueba comparte fincas con el entrenamiento, el desempeño medido incluye lo que el modelo aprende de cada finca en particular, y no solo lo que generaliza.

Ese riesgo es concreto en este histórico, por dos razones:

- Un mismo lote aparece en varios cortes, y los lotes de una hacienda comparten suelo, manejo y distancia al ingenio. En AA1, con una partición aleatoria por filas, el 89.7 % de los registros de prueba pertenecía a un lote que también estaba en entrenamiento.
- `Dist Km` toma un único valor por hacienda. Si una finca está en ambos conjuntos, un modelo flexible puede usar la distancia para reconocerla, en lugar de aprender un efecto de la distancia.

Por eso la partición se agrupa por hacienda: todas las cosechas de una finca quedan en el mismo conjunto. Agrupar por lote no bastaría, porque los demás lotes de la misma hacienda seguirían en el otro conjunto. La validación cruzada de la sección 3.3 usa la misma agrupación.

### 3.1 Partición agrupada por hacienda

`GroupKFold` divide las haciendas en cinco grupos de tamaño parecido en número de registros, así que cada grupo es un conjunto de prueba candidato con cerca del 20 % de los datos. La partición es común a los dos targets: se hace sobre las 21 027 cosechas, y después cada target conserva sus registros válidos.

Entre las cinco candidatas se elige una con una regla fija, que no usa ningún modelo: la que tiene la menor diferencia entre la media del target en prueba y en entrenamiento, medida en desviaciones estándar del entrenamiento, considerando el peor de los dos targets. Cumple el mismo papel que la estratificación en una partición aleatoria, que no se puede aplicar directamente cuando se agrupa.

In [6]:
# 3.1 Partición 80/20 agrupada por hacienda, común a los dos targets
from sklearn.model_selection import GroupKFold

TARGETS = list(FILAS_VALIDAS_POR_TARGET)  # TCH y %Sac.Caña
N_CANDIDATAS = 5                          # cinco grupos de haciendas de ~20 % de los registros cada uno
PROPORCION_TEST_ADMISIBLE = (0.18, 0.22)  # tolerancia alrededor del 20 % pedido


def diferencia_estandarizada(valores_train, valores_test):
    """Diferencia de medias (prueba − entrenamiento) en desviaciones estándar del entrenamiento."""
    return (valores_test.mean() - valores_train.mean()) / valores_train.std()


# Cada pliegue de GroupKFold es un conjunto de prueba candidato; las haciendas no se reparten entre pliegues
particionador = GroupKFold(n_splits=N_CANDIDATAS, shuffle=True, random_state=RANDOM_STATE)
particiones_candidatas = list(particionador.split(df_suertes, groups=df_suertes["Hacienda"]))

filas_candidatas = []
for numero, (posiciones_train, posiciones_test) in enumerate(particiones_candidatas):
    candidata_train, candidata_test = df_suertes.iloc[posiciones_train], df_suertes.iloc[posiciones_test]
    fila = {
        "candidata": numero,
        "proporcion_test": len(candidata_test) / len(df_suertes),
        "haciendas_test": candidata_test["Hacienda"].nunique(),
    }
    for target in TARGETS:
        fila[f"diferencia_{target}"] = diferencia_estandarizada(
            candidata_train.loc[filas_validas(candidata_train, target), target],
            candidata_test.loc[filas_validas(candidata_test, target), target],
        )
    filas_candidatas.append(fila)

tabla_candidatas = pd.DataFrame(filas_candidatas).set_index("candidata")
columnas_diferencia = [f"diferencia_{target}" for target in TARGETS]
tabla_candidatas["peor_diferencia"] = tabla_candidatas[columnas_diferencia].abs().max(axis=1)
print("Particiones candidatas (diferencias en desviaciones estándar del entrenamiento):")
display(tabla_candidatas.round(3))

# Regla de elección: proporción de prueba cercana al 20 % y menor diferencia en el peor de los dos targets
admisibles = tabla_candidatas["proporcion_test"].between(*PROPORCION_TEST_ADMISIBLE)
CANDIDATA_ELEGIDA = tabla_candidatas.loc[admisibles, "peor_diferencia"].idxmin()
posiciones_train, posiciones_test = particiones_candidatas[CANDIDATA_ELEGIDA]
df_train = df_suertes.iloc[posiciones_train]
df_test = df_suertes.iloc[posiciones_test]

# Verificaciones: conjuntos disjuntos que cubren todo el histórico, sin haciendas compartidas
assert len(df_train) + len(df_test) == len(df_suertes)
assert df_train.index.intersection(df_test.index).empty
assert set(df_train["Hacienda"]).isdisjoint(df_test["Hacienda"]), "Hay haciendas en ambos conjuntos"

print(f"\nPartición elegida: candidata {CANDIDATA_ELEGIDA}")
for nombre, conjunto in [("Entrenamiento", df_train), ("Prueba", df_test)]:
    n_lotes_conjunto = conjunto.groupby(["Hacienda", "Suerte"]).ngroups
    print(f"  {nombre}: {formato_miles(len(conjunto))} registros ({len(conjunto) / len(df_suertes):.1%}) | "
          f"haciendas: {formato_miles(conjunto['Hacienda'].nunique())} | lotes: {formato_miles(n_lotes_conjunto)} | "
          + " | ".join(f"válidos {t}: {formato_miles(filas_validas(conjunto, t).sum())}" for t in TARGETS))

Particiones candidatas (diferencias en desviaciones estándar del entrenamiento):


,proporcion_test,haciendas_test,diferencia_TCH,diferencia_%Sac.Caña,peor_diferencia
candidata,,,,,
0,0.206,186,-0.031,0.049,0.049
1,0.204,185,0.090,-0.117,0.117
2,0.217,185,0.008,0.052,0.052
3,0.157,185,-0.058,0.118,0.118
4,0.217,185,-0.018,-0.083,0.083



Partición elegida: candidata 0
  Entrenamiento: 16 689 registros (79.4%) | haciendas: 740 | lotes: 4663 | válidos TCH: 16 407 | válidos %Sac.Caña: 16 299
  Prueba: 4338 registros (20.6%) | haciendas: 186 | lotes: 1136 | válidos TCH: 4265 | válidos %Sac.Caña: 4279


### 3.2 Balance entre entrenamiento y prueba

Al agrupar por hacienda, el conjunto de prueba queda formado por fincas completas, y su composición puede diferir de la del entrenamiento más que en una partición aleatoria. Se comparan los dos conjuntos en dos aspectos:

- **Los targets:** media, desviación estándar y mediana.
- **Las condiciones que más los afectan según el EDA:** zona, sistema de cultivo, tipo de corte, destino semilla, lluvia sin medición y año de cosecha. Las diferencias se expresan en puntos porcentuales.

Una diferencia grande en alguna de ellas no invalida la partición, pero hay que tenerla en cuenta al leer el desempeño en prueba.

In [7]:
# 3.2 Balance de los targets y de las condiciones principales entre entrenamiento y prueba
VARIABLES_BALANCE = ["Zona", "Cultivo", "T.Corte", "Destino 1=Semilla", "lluvia_sin_dato", "Año de cosecha"]


def resumen_target(conjunto, target):
    """Media, desviación estándar y mediana de un target en sus registros válidos."""
    valores = conjunto.loc[filas_validas(conjunto, target), target]
    return pd.Series({"media": valores.mean(), "desviación": valores.std(), "mediana": valores.median()})


def comparar_proporciones(columna, train, test):
    """Porcentaje de cada nivel en entrenamiento y prueba, y su diferencia en puntos porcentuales."""
    tabla = 100 * pd.concat({
        "entrenamiento_%": train[columna].value_counts(normalize=True),
        "prueba_%": test[columna].value_counts(normalize=True),
    }, axis=1).fillna(0)
    tabla["diferencia_pp"] = tabla["prueba_%"] - tabla["entrenamiento_%"]
    return tabla


# Targets
resumen_targets = pd.concat({
    (target, nombre): resumen_target(conjunto, target)
    for target in TARGETS
    for nombre, conjunto in [("entrenamiento", df_train), ("prueba", df_test)]
}, axis=1).T
print("Targets en entrenamiento y prueba:")
display(resumen_targets.round(2))

# Condiciones principales (el año se deriva de Período solo para esta comparación)
train_con_año = df_train.assign(**{"Año de cosecha": df_train["Período"] // 100})
test_con_año = df_test.assign(**{"Año de cosecha": df_test["Período"] // 100})
balance_condiciones = pd.concat(
    {columna: comparar_proporciones(columna, train_con_año, test_con_año) for columna in VARIABLES_BALANCE},
    names=["variable", "nivel"],
)
print("Porcentaje de registros por nivel y diferencia en puntos porcentuales (prueba − entrenamiento):")
display(balance_condiciones.round(1))

mayor_diferencia = balance_condiciones["diferencia_pp"].abs().idxmax()
print(f"Mayor diferencia absoluta: {balance_condiciones['diferencia_pp'].abs().max():.1f} puntos porcentuales "
      f"({mayor_diferencia[0]} = {mayor_diferencia[1]})")

Targets en entrenamiento y prueba:


media  desviación  mediana
TCH       entrenamiento  131.19       31.23   130.27
          prueba         130.22       31.82   129.64
%Sac.Caña entrenamiento   12.31        1.15    12.36
          prueba          12.37        1.14    12.41

Porcentaje de registros por nivel y diferencia en puntos porcentuales (prueba − entrenamiento):


entrenamiento_%  prueba_%  diferencia_pp
variable          nivel                                               
Zona              IP05                   35.5      43.8            8.3
                  IP03                   24.6      24.7            0.0
                  IP06                   19.6      19.4           -0.2
                  IP01                   12.2       6.1           -6.2
                  IP02                    8.0       6.0           -2.0
                  IC07                    0.0       0.0           -0.0
Cultivo           Normal                 96.0      97.9            1.8
                  Organico                4.0       2.1           -1.8
T.Corte           MECANIZADO             80.2      86.2            6.0
                  MANUAL                 17.0      12.3           -4.6
                  NO DEF.                 2.9       1.5           -1.4
                  SEMILLA                 0.0       0.0           -0.0
Destino 1=Semilla 0                      98.4      99.6            1.2
                  1                       1.6       0.4           -1.2
lluvia_sin_dato   0                      65.8      53.0          -12.8
                  1                      34.2      47.0           12.8
Año de cosecha    2023                   14.4      14.4            0.0
                  2018                   13.8      13.1           -0.8
                  2022                   13.8      14.6            0.8
                  2019                   13.6      13.3           -0.4
                  2020                   13.6      14.2            0.5
                  2017                   12.2      11.0           -1.2
                  2021                   11.7      12.7            1.0
                  2024                    6.8       6.8           -0.0

Mayor diferencia absoluta: 12.8 puntos porcentuales (lluvia_sin_dato = 0)


### 3.3 Pliegues de validación cruzada

Los hiperparámetros y la comparación entre modelos se deciden con validación cruzada de 5 pliegues sobre el entrenamiento, agrupada por hacienda igual que la partición. Así, cada pliegue de validación está formado por fincas que el modelo no vio en ese ajuste.

Los pliegues se asignan una sola vez, por hacienda, y se guardan. Todos los modelos de las secciones 4 y 5, en los dos targets, se evalúan exactamente en las mismas particiones del entrenamiento. Eso permite comparar los modelos pliegue a pliegue y leer sus diferencias frente a la variación entre pliegues. Para cada target se construye un objeto con sus conjuntos de entrenamiento y prueba y sus pliegues.

In [8]:
# 3.3 Pliegues de validación cruzada agrupados por hacienda, fijos para todos los modelos
from dataclasses import dataclass

from sklearn.model_selection import PredefinedSplit

N_PLIEGUES_CV = 5


@dataclass
class DatosTarget:
    """Conjuntos de entrenamiento y prueba de un target, con sus pliegues de validación cruzada."""
    X_train: pd.DataFrame
    y_train: pd.Series
    X_test: pd.DataFrame
    y_test: pd.Series
    haciendas_train: pd.Series   # hacienda de cada registro de entrenamiento
    pliegues: PredefinedSplit    # validación cruzada común a todos los modelos


# Asignación de cada registro de entrenamiento a un pliegue, por hacienda
asignador_pliegues = GroupKFold(n_splits=N_PLIEGUES_CV, shuffle=True, random_state=RANDOM_STATE)
pliegue_por_registro = pd.Series(-1, index=df_train.index, name="pliegue_cv")
for numero_pliegue, (_, posiciones_validacion) in enumerate(
        asignador_pliegues.split(df_train, groups=df_train["Hacienda"])):
    pliegue_por_registro.iloc[posiciones_validacion] = numero_pliegue
assert (pliegue_por_registro >= 0).all(), "Hay registros sin pliegue asignado"

# Conjuntos de cada target; los pliegues se toman de la asignación común
datos_por_target = {}
for target in TARGETS:
    X_entrenamiento, y_entrenamiento = separar_target(df_train, target)
    X_prueba, y_prueba = separar_target(df_test, target)
    datos_por_target[target] = DatosTarget(
        X_train=X_entrenamiento,
        y_train=y_entrenamiento,
        X_test=X_prueba,
        y_test=y_prueba,
        haciendas_train=df_train.loc[X_entrenamiento.index, "Hacienda"],
        pliegues=PredefinedSplit(pliegue_por_registro.loc[X_entrenamiento.index].to_numpy()),
    )

# Verificación: en cada pliegue, las haciendas de validación no aparecen en el ajuste
filas_resumen = []
for target, datos in datos_por_target.items():
    for numero_pliegue, (posiciones_ajuste, posiciones_validacion) in enumerate(datos.pliegues.split()):
        haciendas_ajuste = set(datos.haciendas_train.iloc[posiciones_ajuste])
        haciendas_validacion = set(datos.haciendas_train.iloc[posiciones_validacion])
        assert haciendas_ajuste.isdisjoint(haciendas_validacion), f"{target}: haciendas compartidas en el pliegue {numero_pliegue}"
        filas_resumen.append({
            "target": target,
            "pliegue": numero_pliegue,
            "registros_validacion": len(posiciones_validacion),
            "haciendas_validacion": len(haciendas_validacion),
            "media_target_validacion": datos.y_train.iloc[posiciones_validacion].mean(),
        })

print("Pliegues de validación cruzada (dentro del entrenamiento):")
display(pd.DataFrame(filas_resumen).set_index(["target", "pliegue"]).round(2))
for target, datos in datos_por_target.items():
    print(f"{target}: entrenamiento {formato_miles(len(datos.y_train))} | prueba {formato_miles(len(datos.y_test))}")

Pliegues de validación cruzada (dentro del entrenamiento):


registros_validacion  haciendas_validacion  \
target    pliegue                                               
TCH       0                        3446                   148   
          1                        3012                   145   
          2                        4042                   147   
          3                        2738                   146   
          4                        3169                   144   
%Sac.Caña 0                        3402                   148   
          1                        2977                   148   
          2                        4057                   148   
          3                        2772                   148   
          4                        3091                   148   

                   media_target_validacion  
target    pliegue                           
TCH       0                         130.16  
          1                         133.35  
          2                         128.58  
          3                         132.31  
          4                         132.60  
%Sac.Caña 0                          12.33  
          1                          12.33  
          2                          12.23  
          3                          12.35  
          4                          12.32

TCH: entrenamiento 16 407 | prueba 4265
%Sac.Caña: entrenamiento 16 299 | prueba 4279


### Conclusión de la sección 3

**Partición.** De las cinco particiones candidatas, la elegida es la que menos desplaza la media de los targets entre entrenamiento y prueba: 0.049 desviaciones estándar en el peor de los dos. El entrenamiento tiene 16 689 registros (79.4 %) de 740 haciendas y 4663 lotes; la prueba, 4338 registros (20.6 %) de 186 haciendas y 1136 lotes. Ninguna hacienda aparece en ambos conjuntos. Para `TCH` quedan 16 407 registros de entrenamiento y 4265 de prueba; para `%Sac.Caña`, 16 299 y 4279.

**Balance.** Los targets son prácticamente iguales en los dos conjuntos: el TCH medio es de 131.2 t/ha en entrenamiento y 130.2 en prueba, y la sacarosa de 12.31 % y 12.37 %, con desviaciones casi idénticas. Los años de cosecha también están balanceados, con diferencias de 1.2 puntos porcentuales o menos. Las diferencias aparecen en condiciones que dependen de la finca:

- **Lluvia sin medición:** el 47.0 % de los registros de prueba no tiene lluvia registrada, frente al 34.2 % del entrenamiento. Como los años están balanceados, la diferencia no viene del periodo sino de las fincas que quedaron en cada conjunto.
- **Zona:** la prueba tiene más registros de IP05 (43.8 % frente a 35.5 %) y menos de IP01 (6.1 % frente a 12.2 %).
- **Manejo:** en prueba hay más cosecha mecanizada (86.2 % frente a 80.2 %), menos lotes orgánicos (2.1 % frente a 4.0 %) y menos lotes de semilla (0.4 % frente a 1.6 %).

Estas diferencias son la consecuencia esperada de asignar fincas completas a cada conjunto, y la partición se mantiene: la regla de elección se fijó antes de ver el balance. No afectan la comparación entre modelos, que se evalúan todos en el mismo conjunto de prueba. Pero el error en prueba puede ser mayor que en validación cruzada, porque en casi la mitad de esos registros la lluvia no aporta información. En la sección 7 el error se reporta por separado para registros con y sin lluvia medida. Los lotes orgánicos y de semilla de prueba son pocos, así que el error en esos subgrupos se estimará con poca precisión.

**Validación cruzada.** Los cinco pliegues del entrenamiento tienen entre 144 y 148 haciendas cada uno, y entre 2738 y 4042 registros, porque las fincas tienen tamaños distintos. La media del target por pliegue va de 128.6 a 133.4 t/ha en `TCH` y de 12.23 % a 12.35 % en `%Sac.Caña`. En ningún pliegue se comparten haciendas entre ajuste y validación. Estos mismos pliegues se usan en todos los modelos de las secciones 4 y 5.